In [ ]:
import numpy as np
import torch
import torch.nn.functional as F
import time
import matplotlib.pyplot as plt

import sys, os
sys.path.append(os.path.abspath("../nca3d"))

from nca3d.core import *
import nca3d.viz as viz
import nca3d.io as io

filepath = os.path.abspath("../assets/donut/model.obj")
device = "cuda" if torch.cuda.is_available() else "cpu"
print("device ->", device)

target = io.obj_to_tensor(filepath=filepath, grid_size=(32, 32, 32), mode="rgba").to(device).float()
viz.show_volume_alpha_mpl(target, point_size=100)
viz.show_volume_color_pv(target, notebook=True, point_size=30)

In [ ]:
cell_cfg = CellConfig(hidden_channels=8, visible_channels=4, alive_threshold=0.01)
perc_cfg = PerceptionConfig()
upd_cfg  = UpdateConfig(hidden_dim=64, stochastic_update=False, fire_rate=0.5)
grid_cfg = GridConfig(size=(32, 32, 32))
model = Grid3D(cell_cfg, perc_cfg, upd_cfg, grid_cfg).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=2e-3)

for iter in range(2000):
    optimizer.zero_grad()
    state = model.seed_center(1, device)
    c = 16
    state[:, :8, c:c+1, c:c+1, c:c+1] = torch.randn(1, 8, 1, 1, 1, device=device) * 0.05
    
    state = model(state, steps=64 + iter//100)
    
    # Extract RGBA predictions and targets
    rgba_pred   = state[:, -4:]
    rgba_target = target[:, :4]
    
    # Compute loss on RGBA
    loss = F.mse_loss(rgba_pred[:, :, 10:22, 10:22, 10:22],
                      rgba_target[:, :, 10:22, 10:22, 10:22])
    
    loss.backward()
    optimizer.step()
    
    if iter % 200 == 0:
        # Count alive voxels
        alpha = state[:, -1]  # Extract alpha channel
        alive_voxels = (alpha > cell_cfg.alive_threshold).sum().item()
        total_voxels = alpha.numel()
        alive_percentage = 100.0 * alive_voxels / total_voxels
        
        alive_mask = model.cell.alive_mask
        alive_mask_count = alive_mask.sum().item() if alive_mask.numel() > 1 else 0
        
        print(f"{iter:05d} | loss={loss.item():.6f} | "
              f"alive: {alive_voxels}/{total_voxels} ({alive_percentage:.1f}%) | "
              f"alive_mask: {alive_mask_count}")
        
        viz.show_slice_color_comparison_mpl(state, target, visible_channels=4, axis=1, idx=15)

In [ ]:
viz.show_volume_color_comparison_pv(state, target, visible_channels=4, notebook=True, threshold=0.9)

In [ ]:
viz.show_volume_alpha_comparison_pv(state, target, visible_channels=4, notebook=True, threshold=0.9)